In [ ]:
# ============================================================
# CELL 1 — Configure AWS and Verify Connections
# ============================================================
import os, subprocess
from google.colab import userdata

# Load credentials from Colab Secrets
AWS_KEY     = userdata.get('AWS_ACCESS_KEY_ID')
AWS_SECRET  = userdata.get('AWS_SECRET_ACCESS_KEY')
DB_PASSWORD = userdata.get('DB_PASSWORD')

DB_HOST          = 'geoai-mlops-p1-postgres.cqbao8c24eu5.us-east-1.rds.amazonaws.com'
DB_USER          = 'geoai_admin'
DB_NAME_FEATURES = 'geoai_features'
S3_BUCKET        = 'geoai-mlops-p1-data-288528696055'
MLFLOW_URL       = 'http://3.91.55.220:5000'

# Set in os.environ for all subsequent cells
os.environ['AWS_ACCESS_KEY_ID']     = AWS_KEY
os.environ['AWS_SECRET_ACCESS_KEY'] = AWS_SECRET
os.environ['AWS_DEFAULT_REGION']    = 'us-east-1'

# Install boto3 in system Python for verification
os.system('pip install -q boto3')

# Verify S3
import boto3
s3   = boto3.client('s3', region_name='us-east-1')
resp = s3.list_objects_v2(
    Bucket=S3_BUCKET, Prefix='processed/patches/', MaxKeys=1)
print(f'✅ S3 connected — objects: {resp["KeyCount"]}')

# Verify GPU
r = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
    capture_output=True, text=True
)
if r.returncode == 0:
    print(f'✅ GPU: {r.stdout.strip()}')
else:
    print('⚠️  No GPU detected')

print()
print(f'   AWS_KEY:   ...{AWS_KEY[-4:]}')
print(f'   S3_BUCKET: {S3_BUCKET}')
print(f'   DB_HOST:   {DB_HOST}')
print(f'   MLflow:    {MLFLOW_URL}')
print()
print('✅ Cell 1 complete — AWS configured')



In [ ]:
# ============================================================
# CELL 2 — Setup Virtual Environment
# ============================================================
import subprocess, os

VENV_PYTHON = '/content/venv/bin/python3'
VENV_PIP    = '/content/venv/bin/pip'

# System dependencies
os.system('apt-get install -y libgeos-dev libgdal-dev -q')
os.environ["MPLBACKEND"] = "agg"

# Create venv
os.system('python3 -m venv /content/venv')
os.system('curl -sS https://bootstrap.pypa.io/get-pip.py | /content/venv/bin/python3')

def install(packages):
    r = subprocess.run(
        [VENV_PIP, 'install', '-q'] + packages,
        capture_output=True, text=True
    )
    if r.returncode != 0:
        print(f'Error: {r.stderr[-500:]}')
        return False
    return True

print('Step 1: terratorch (first — owns its dependencies)...')
install(['terratorch'])

print('Step 2: torch (force reinstall after terratorch)...')
install(['--force-reinstall', 'torch==2.2.0', 'torchvision==0.17.0'])

print('Step 3: dependencies...')
install([
    'transformers==4.40.0', 'timm', 'einops',
    'mlflow==2.14.3', 'psycopg2-binary', 'boto3',
    'rasterio', 'scipy', 'pystac-client',
    'huggingface_hub==0.20.3', 'awscli', 'pyyaml',
    'pandas', 'tqdm',
])

print('Step 4: pin all critical versions last...')
install([
    'numpy==1.26.4',
    'huggingface-hub==0.20.3',
    'protobuf==3.20.3',
    'setuptools==69.5.1',
])

os.system('pip install -q boto3 rasterio pandas')
print('✅ boto3 + rasterio + pandas installed in system Python')

r = subprocess.run([VENV_PYTHON, '-c', '''
import os; os.environ["MPLBACKEND"] = "agg"
import numpy as np, torch, pkg_resources
import google.protobuf
from terratorch.registry import BACKBONE_REGISTRY
print(f"numpy:      {np.__version__}")
print(f"torch:      {torch.__version__}")
print(f"protobuf:   {google.protobuf.__version__}")
print(f"setuptools: {pkg_resources.get_distribution('setuptools').version}")
print("terratorch: OK")
'''], capture_output=True, text=True)
print(r.stdout)
if r.stderr: print('STDERR:', r.stderr[-200:])
print('✅ Cell 2 complete — venv ready')



In [ ]:
# ============================================================
# CELL 3 — Download Repo + Pairs + Stats + Features Cache
# Pairs, stats and features saved to Google Drive (skip if exist)
# ============================================================
import subprocess, os, boto3, tarfile
from google.colab import drive, userdata

VENV_PYTHON = '/content/venv/bin/python3'
S3_BUCKET   = 'geoai-mlops-p1-data-288528696055'
AWS_KEY     = userdata.get('AWS_ACCESS_KEY_ID')
AWS_SECRET  = userdata.get('AWS_SECRET_ACCESS_KEY')

os.environ['AWS_ACCESS_KEY_ID']     = AWS_KEY
os.environ['AWS_SECRET_ACCESS_KEY'] = AWS_SECRET
os.environ['AWS_DEFAULT_REGION']    = 'us-east-1'

# Mount Google Drive
drive.mount('/gdrive')

# Create Google Drive directories
os.makedirs('/gdrive/MyDrive/geoai_mlops/pairs',             exist_ok=True)
os.makedirs('/gdrive/MyDrive/geoai_mlops/stats',             exist_ok=True)
os.makedirs('/gdrive/MyDrive/geoai_mlops/patches',           exist_ok=True)
os.makedirs('/gdrive/MyDrive/geoai_mlops/features',          exist_ok=True)
os.makedirs('/gdrive/MyDrive/geoai_mlops/checkpoints/local', exist_ok=True)
os.makedirs('/gdrive/MyDrive/geoai_mlops/checkpoints/best',  exist_ok=True)
print('✅ Google Drive directories ready')

s3 = boto3.client('s3', region_name='us-east-1')

# Download and extract repo to /content (always fresh — small file)
print('\nDownloading repo from S3...')
s3.download_file(S3_BUCKET, 'repo/geoai-mlops-p1-colab-fixes.tar.gz',
                 '/tmp/geoai-mlops-p1.tar.gz')
size = os.path.getsize('/tmp/geoai-mlops-p1.tar.gz')
print(f'Downloaded: {size/1e6:.1f}MB')

with tarfile.open('/tmp/geoai-mlops-p1.tar.gz', 'r:gz') as tar:
    tar.extractall('/content/')
print('✅ Repo extracted')
print('   Training files:', os.listdir('/content/geoai-mlops-p1/src/training/'))

# Create /content data dirs
os.makedirs('/content/geoai-mlops-p1/data/pairs',             exist_ok=True)
os.makedirs('/content/geoai-mlops-p1/data/stats',             exist_ok=True)
os.makedirs('/content/geoai-mlops-p1/data/checkpoints/local', exist_ok=True)

# Download pairs → Google Drive (skip if already exist)
print('\nDownloading pairs to Google Drive...')
for f in ['train_pairs.csv', 'val_pairs.csv', 'test_pairs.csv']:
    path = f'/gdrive/MyDrive/geoai_mlops/pairs/{f}'
    if os.path.exists(path):
        print(f'  ✅ {f} already exists — skipping')
    else:
        s3.download_file(S3_BUCKET, f'training/pairs/{f}', path)
        print(f'  ✅ {f} downloaded')

# Download stats → Google Drive (skip if already exist)
print('\nDownloading stats to Google Drive...')
for f in ['tabular_stats.json', 'band_stats.json',
          'day_gap_stats.json', 'stats_summary.json']:
    path = f'/gdrive/MyDrive/geoai_mlops/stats/{f}'
    if os.path.exists(path):
        print(f'  ✅ {f} already exists — skipping')
    else:
        s3.download_file(S3_BUCKET, f'training/stats/{f}', path)
        print(f'  ✅ {f} downloaded')

# Download features cache → Google Drive (skip if exists)
print('\nDownloading features cache to Google Drive...')
path = '/gdrive/MyDrive/geoai_mlops/features/features_cache.parquet'
if os.path.exists(path):
    size = os.path.getsize(path)
    print(f'  ✅ features_cache.parquet already exists ({size/1e6:.1f}MB) — skipping')
else:
    s3.download_file(
        S3_BUCKET,
        'training/features/features_cache.parquet',
        path
    )
    size = os.path.getsize(path)
    print(f'  ✅ features_cache.parquet downloaded ({size/1e6:.1f}MB)')

print()
print('✅ Cell 3 complete')



In [ ]:
# ============================================================
# CELL 4 — Download Patches to Google Drive (PARALLEL)
# Uses 10 concurrent threads — 5-10x faster than sequential
# ============================================================
!pip install -q boto3

import os, boto3, time
from google.colab import userdata, drive
from concurrent.futures import ThreadPoolExecutor, as_completed

drive.mount('/gdrive')
os.makedirs('/gdrive/MyDrive/geoai_mlops/patches', exist_ok=True)

AWS_KEY    = userdata.get('AWS_ACCESS_KEY_ID')
AWS_SECRET = userdata.get('AWS_SECRET_ACCESS_KEY')

os.environ['AWS_ACCESS_KEY_ID']     = AWS_KEY
os.environ['AWS_SECRET_ACCESS_KEY'] = AWS_SECRET
os.environ['AWS_DEFAULT_REGION']    = 'us-east-1'

BUCKET  = 'geoai-mlops-p1-data-288528696055'
PREFIX  = 'processed/patches/'
LOCAL   = '/gdrive/MyDrive/geoai_mlops/patches/'
WORKERS = 10  # parallel threads

# List all files
s3 = boto3.client('s3', region_name='us-east-1')
print('Listing files in S3...')
paginator = s3.get_paginator('list_objects_v2')
all_keys  = []
for page in paginator.paginate(Bucket=BUCKET, Prefix=PREFIX):
    for obj in page.get('Contents', []):
        all_keys.append(obj['Key'])
print(f'Found {len(all_keys):,} files — downloading with {WORKERS} threads')

# Download function (each thread gets its own S3 client)
def download_file(key):
    client     = boto3.client('s3', region_name='us-east-1')
    local_path = os.path.join(LOCAL, key[len(PREFIX):])
    os.makedirs(os.path.dirname(local_path), exist_ok=True)
    # Skip if already downloaded
    if os.path.exists(local_path):
        return True
    try:
        client.download_file(BUCKET, key, local_path)
        return True
    except Exception:
        return False

# Run parallel downloads
errors    = 0
completed = 0
start     = time.time()

with ThreadPoolExecutor(max_workers=WORKERS) as executor:
    futures = {executor.submit(download_file, key): key
               for key in all_keys}
    for future in as_completed(futures):
        completed += 1
        if not future.result():
            errors += 1
        if completed % 1000 == 0:
            elapsed = time.time() - start
            pct     = completed / len(all_keys) * 100
            eta     = (elapsed / completed) * (len(all_keys) - completed)
            print(f'  {pct:.1f}% — {completed:,}/{len(all_keys):,} | '
                  f'elapsed: {elapsed/60:.1f}min | ETA: {eta/60:.1f}min')

elapsed = time.time() - start
print(f'✅ Done in {elapsed/60:.1f} minutes — errors: {errors}')
print('Patches saved to Google Drive ✅')


In [ ]:
# ============================================================
# CELL 5 — Write Training Config (Colab overrides)
# Loads repo train_config.json and updates Colab-specific paths
# ============================================================
import json, os, math, pandas as pd
from google.colab import userdata

DB_PASSWORD = userdata.get('DB_PASSWORD')
DB_HOST     = 'geoai-mlops-p1-postgres.cqbao8c24eu5.us-east-1.rds.amazonaws.com'
DB_USER     = 'geoai_admin'
DB_NAME     = 'geoai_features'

# Ensure Google Drive directories exist
os.makedirs('/gdrive/MyDrive/geoai_mlops/pairs',             exist_ok=True)
os.makedirs('/gdrive/MyDrive/geoai_mlops/stats',             exist_ok=True)
os.makedirs('/gdrive/MyDrive/geoai_mlops/patches',           exist_ok=True)
os.makedirs('/gdrive/MyDrive/geoai_mlops/features',          exist_ok=True)
os.makedirs('/gdrive/MyDrive/geoai_mlops/checkpoints/local', exist_ok=True)
os.makedirs('/gdrive/MyDrive/geoai_mlops/checkpoints/best',  exist_ok=True)
print('✅ Google Drive directories ready')

# ── Incremental training settings ────────────────────────────
CHUNK_NUMBER = 6      # ← ONLY change this each session
CHUNK_SIZE   = 26512  # pairs per chunk — never change

# Auto-compute offset and total chunks
train_df     = pd.read_csv('/gdrive/MyDrive/geoai_mlops/pairs/train_pairs.csv')
TOTAL_PAIRS  = len(train_df)
PAIRS_OFFSET = (CHUNK_NUMBER - 1) * CHUNK_SIZE
PAIRS_END    = min(PAIRS_OFFSET + CHUNK_SIZE, TOTAL_PAIRS)
TOTAL_CHUNKS = math.ceil(TOTAL_PAIRS / CHUNK_SIZE)
IS_LAST      = PAIRS_END >= TOTAL_PAIRS

print(f'\nIncremental training plan:')
print(f'  Total train pairs:  {TOTAL_PAIRS:,}')
print(f'  Chunk size:         {CHUNK_SIZE:,}')
print(f'  Total chunks:       {TOTAL_CHUNKS}')
print(f'  Current chunk:      {CHUNK_NUMBER} / {TOTAL_CHUNKS}')
print(f'  Pairs this chunk:   {PAIRS_OFFSET:,} → {PAIRS_END:,} ({PAIRS_END-PAIRS_OFFSET:,} pairs)')
print(f'  {"⚠️  LAST CHUNK" if IS_LAST else "More chunks after this"}')

if PAIRS_OFFSET >= TOTAL_PAIRS:
    print('\n✅ All data already trained — no more chunks!')
    raise SystemExit('Training complete — all chunks done')

# Load repo config as base
config_path = '/content/geoai-mlops-p1/src/training/train_config.json'
with open(config_path) as f:
    config = json.load(f)

# Override with Colab-specific paths and credentials
config.update({
    'project_dir':          '/content/geoai-mlops-p1',

    # Pairs and stats — Google Drive
    'train_pairs_csv':      '/gdrive/MyDrive/geoai_mlops/pairs/train_pairs.csv',
    'val_pairs_csv':        '/gdrive/MyDrive/geoai_mlops/pairs/val_pairs.csv',
    'test_pairs_csv':       '/gdrive/MyDrive/geoai_mlops/pairs/test_pairs.csv',
    'tabular_stats':        '/gdrive/MyDrive/geoai_mlops/stats/tabular_stats.json',
    'band_stats':           '/gdrive/MyDrive/geoai_mlops/stats/band_stats.json',
    'day_gap_stats':        '/gdrive/MyDrive/geoai_mlops/stats/day_gap_stats.json',

    # Patches — Google Drive (Cell 5b updates to /content/patches/)
    'patches_local_dir':    '/gdrive/MyDrive/geoai_mlops/patches',

    # Features cache
    'features_cache':       '/gdrive/MyDrive/geoai_mlops/features/features_cache.parquet',

    # Checkpoints — Google Drive
    'checkpoint_local_dir':  '/gdrive/MyDrive/geoai_mlops/checkpoints/local',
    'checkpoint_best_dir':   '/gdrive/MyDrive/geoai_mlops/checkpoints/best',
    'checkpoint_gdrive_steps': 200,

    # DB credentials
    'db_host': DB_HOST,
    'db_user': DB_USER,
    'db_pass': DB_PASSWORD,
    'db_name': DB_NAME,

    # MLflow — Colab points to EC2
    'mlflow_tracking_uri': 'http://3.91.55.220:5000',

    # Colab override — 2 workers max (Colab has 2 CPUs)
    'num_workers': 2,

    # In Cell 5

    'batch_size':  64,   # was 32 — larger batches = more GPU work per step

    # Colab epoch override — keep short for session time limits
    'phase1_epochs': 3,
    'phase2_epochs': 5,


    # Incremental training
    'chunk_number':    CHUNK_NUMBER,
    'chunk_size':      CHUNK_SIZE,
    'pairs_offset':    PAIRS_OFFSET,
    'max_train_pairs': PAIRS_END - PAIRS_OFFSET,  # pairs this chunk
    'max_val_pairs':   len(pd.read_csv('/gdrive/MyDrive/geoai_mlops/pairs/val_pairs.csv')),
    'max_test_pairs':  len(pd.read_csv('/gdrive/MyDrive/geoai_mlops/pairs/test_pairs.csv')),
    'total_chunks':    TOTAL_CHUNKS,
    'is_last_chunk':   IS_LAST,

})

# Write updated config back
with open(config_path, 'w') as f:
    json.dump(config, f, indent=2)

print('\n✅ Config written')
print()
print('Training settings:')
print(f'  batch_size:            {config["batch_size"]}')
print(f'  phase1_epochs:         {config["phase1_epochs"]}')
print(f'  phase2_epochs:         {config["phase2_epochs"]}')
print(f'  early_stopping:        patience={config["early_stopping_patience"]}')
print(f'  num_workers:           {config["num_workers"]}')
print(f'  batch_size:            {config["batch_size"]}')
print(f'  chunk_number:          {config["chunk_number"]} / {config["total_chunks"]}')
print(f'  pairs_offset:          {config["pairs_offset"]:,}')
print(f'  max_train_pairs:       {config["max_train_pairs"]:,}')
print(f'  max_val_pairs:         {config["max_val_pairs"]:,}')
print(f'  max_test_pairs:        {config["max_test_pairs"]:,}')
print(f'  MLflow:                {config["mlflow_tracking_uri"]}')
print()
print('Note: Cell 5b updates patches_local_dir → /content/patches/')

In [ ]:
# ============================================================
# CELL 5b — Smart copy — only patches needed for training
# Parallel copy with 32 threads — faster than sequential
# Must run AFTER Cell 5 (reads config + pair CSVs)
# ============================================================
import os, time, json, shutil, rasterio, numpy as np, pandas as pd
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed

# Read config
cfg       = json.load(open('/content/geoai-mlops-p1/src/training/train_config.json'))
GDRIVE    = '/gdrive/MyDrive/geoai_mlops/patches'
LOCAL_SSD = '/content/patches/'

print(f'Source:      {GDRIVE}')
print(f'Destination: {LOCAL_SSD}')
print(f'Chunk:       {cfg["chunk_number"]} / {cfg["total_chunks"]}')
print(f'Train pairs: {cfg["pairs_offset"]:,} → {cfg["pairs_offset"]+cfg["max_train_pairs"]:,}')

# Step 1: Read pair CSVs and collect needed patch IDs
print('\nReading pair CSVs to find needed patches...')
needed_patches = set()

# Train pairs — deterministic slice
train_df = pd.read_csv(cfg['train_pairs_csv'])
offset   = cfg.get('pairs_offset', 0)
max_n    = cfg.get('max_train_pairs', len(train_df))
train_df = train_df.iloc[offset:offset + max_n]
needed_patches.update(train_df['patch_id_1'].tolist())
needed_patches.update(train_df['patch_id_2'].tolist())
print(f'  train: {len(train_df):,} pairs (rows {offset:,}→{offset+len(train_df):,})')

# Val pairs — ALL
val_df = pd.read_csv(cfg['val_pairs_csv'])
needed_patches.update(val_df['patch_id_1'].tolist())
needed_patches.update(val_df['patch_id_2'].tolist())
print(f'  val:   {len(val_df):,} pairs (ALL)')

# Test pairs — ALL
test_df = pd.read_csv(cfg['test_pairs_csv'])
needed_patches.update(test_df['patch_id_1'].tolist())
needed_patches.update(test_df['patch_id_2'].tolist())
print(f'  test:  {len(test_df):,} pairs (ALL)')

print(f'Unique patches needed: {len(needed_patches):,}')

# Step 2: Build file list — check which files are missing
print('\nChecking which files need copying...')

def get_patch_files(patch_id):
    parts      = patch_id.split('__')
    tile_id    = parts[0]
    patch_name = parts[1] if len(parts) > 1 else patch_id
    patch_dir  = os.path.join(GDRIVE, tile_id, patch_name)
    results    = []
    try:
        with os.scandir(patch_dir) as entries:
            for entry in entries:
                if entry.is_file():
                    dst = os.path.join(LOCAL_SSD, tile_id,
                                       patch_name, entry.name)
                    # Only add if destination doesn't exist yet
                    if not os.path.exists(dst):
                        results.append((entry.path, dst))
    except FileNotFoundError:
        pass
    return results

files_to_copy = []
missing       = 0

with tqdm(total=len(needed_patches), unit='patch',
          desc='Checking files', ncols=80) as pbar:
    with ThreadPoolExecutor(max_workers=16) as executor:
        futures = {executor.submit(get_patch_files, pid): pid
                   for pid in needed_patches}
        for future in as_completed(futures):
            result = future.result()
            if result:
                files_to_copy.extend(result)
            else:
                missing += 1
            pbar.update(1)

print(f'Files to copy:      {len(files_to_copy):,}')
print(f'Already on SSD:     {missing + len(needed_patches) - len(files_to_copy) // 5:,} patches')
print(f'Missing patch dirs: {missing}')

if len(files_to_copy) == 0:
    print('✅ All patches already on local SSD — skipping copy')
else:
    print(f'\nCopying {len(files_to_copy):,} missing files...')
    os.makedirs(LOCAL_SSD, exist_ok=True)

    # Step 3: Parallel copy with 32 threads
    WORKERS = 32

    def copy_file(args):
        src, dst = args
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        try:
            shutil.copy2(src, dst)
            return True
        except Exception:
            return False

    errors = 0
    start  = time.time()

    with tqdm(total=len(files_to_copy), unit='file',
              desc='Copying to SSD', ncols=80,
              bar_format='{l_bar}{bar}| {n_fmt}/{total_fmt} '
                         '[{elapsed}<{remaining}, {rate_fmt}]') as pbar:
        with ThreadPoolExecutor(max_workers=WORKERS) as executor:
            futures = {executor.submit(copy_file, args): args
                       for args in files_to_copy}
            for future in as_completed(futures):
                if not future.result():
                    errors += 1
                pbar.update(1)

    elapsed = time.time() - start
    count   = sum(len(f) for _,_,f in os.walk(LOCAL_SSD))
    print(f'\n✅ Done in {elapsed/60:.1f} minutes')
    print(f'   {count:,} total files on SSD — errors: {errors}')

# Always update config to point to local SSD
cfg['patches_local_dir'] = LOCAL_SSD
with open('/content/geoai-mlops-p1/src/training/train_config.json', 'w') as f:
    json.dump(cfg, f, indent=2)
print(f'\n✅ Config updated — patches_local_dir → {LOCAL_SSD}')

# Verify speed
print('\nVerifying local SSD read speed...')
files = []
for root, dirs, fs in os.walk(LOCAL_SSD):
    for f in fs:
        if f.endswith('.tif'):
            files.append(os.path.join(root, f))
    if len(files) >= 10:
        break

start = time.time()
for f in files[:10]:
    with rasterio.open(f) as src:
        arr = src.read(1)
elapsed = time.time() - start
print(f'10 files:          {elapsed:.3f}s = {elapsed/10*1000:.1f}ms per file')
print(f'Per epoch (train): {elapsed * cfg["max_train_pairs"] / 3600:.2f} hours')
print(f'Per epoch (val):   {elapsed * cfg["max_val_pairs"] / 3600:.2f} hours')
print()
print('✅ Cell 5b complete — ready for Cell 6')

In [ ]:
# ============================================================
# CELL 6 — Verify Prithvi-EO-1.0-100M Loads Correctly
# Confirms TerraTorch + encoder working before training
# ============================================================
import subprocess
from google.colab import userdata

VENV_PYTHON = '/content/venv/bin/python3'
AWS_KEY     = userdata.get('AWS_ACCESS_KEY_ID')
AWS_SECRET  = userdata.get('AWS_SECRET_ACCESS_KEY')

r = subprocess.run([VENV_PYTHON, '-c', f'''
import os
os.environ["MPLBACKEND"]            = "agg"
os.environ["AWS_ACCESS_KEY_ID"]     = "{AWS_KEY}"
os.environ["AWS_SECRET_ACCESS_KEY"] = "{AWS_SECRET}"
os.environ["AWS_DEFAULT_REGION"]    = "us-east-1"

import torch
from terratorch.registry import BACKBONE_REGISTRY

print("Loading Prithvi-EO-1.0-100M via TerraTorch...")
encoder = BACKBONE_REGISTRY.build(
    "prithvi_eo_v1_100",
    pretrained=True,
    num_frames=1,
    in_chans=6,
)
params = sum(p.numel() for p in encoder.parameters())
print(f"✅ Encoder loaded: {{params/1e6:.0f}}M parameters")

# Test forward pass
x = torch.zeros(1, 6, 224, 224)
with torch.no_grad():
    out = encoder(x)

# Extract embedding — last layer, mean pool patch tokens
embedding = out[-1][:, 1:, :].mean(dim=1)
print(f"✅ Embedding shape: {{embedding.shape}}")
assert embedding.shape == torch.Size([1, 768]), f"Expected [1,768] got {{embedding.shape}}"

# Test on GPU if available
if torch.cuda.is_available():
    encoder = encoder.cuda()
    with torch.no_grad():
        out_gpu = encoder(x.cuda())
    emb_gpu = out_gpu[-1][:, 1:, :].mean(dim=1)
    print(f"✅ GPU forward pass: {{emb_gpu.shape}}")
    print(f"   GPU: {{torch.cuda.get_device_name(0)}}")
else:
    print("⚠️  No GPU detected — check runtime type")

del encoder
torch.cuda.empty_cache() if torch.cuda.is_available() else None
print("✅ Cell 6 complete — Prithvi ready for training")
'''], capture_output=True, text=True)

print(r.stdout)
if r.stderr: print('STDERR:', r.stderr[-300:])





In [ ]:
# ============================================================
# CELL 7 — Verify Training Scripts + Add Progress Display
#           + Wire up --resume + versioned checkpoints
#           + Fix dataset pairs_offset (train only)
#           + Metrics log after each chunk
#           + Crash recovery (resume from latest checkpoint)
#           + Robust checkpoint format handling (both formats)
#           + Fix best model loading for test evaluation
#           + Best model path recovery when training skipped
#           + tqdm + GPU cache clear for test evaluation
# ============================================================
import os, sys, subprocess, re

dataset_path = '/content/geoai-mlops-p1/src/training/dataset.py'
train_path   = '/content/geoai-mlops-p1/src/training/train.py'
model_path   = '/content/geoai-mlops-p1/src/training/model.py'

ds    = open(dataset_path).read()
train = open(train_path).read()
model = open(model_path).read()

print('Verification:')
print(f'  patches_local_dir in dataset:  {"✅" if "patches_local_dir" in ds else "❌"}')
print(f'  features_cache in dataset:     {"✅" if "features_cache" in ds else "❌"}')
print(f'  _tabular_mean in dataset:      {"✅" if "_tabular_mean" in ds else "❌"}')
print(f'  _compute_target in dataset:    {"✅" if "ndvi_mean" in ds else "❌"}')
print(f'  skipping PostGIS in dataset:   {"✅" if "skipping PostGIS" in ds else "❌"}')
print(f'  features_cache in train:       {"✅" if "features_cache" in train else "❌"}')
print(f'  torch.cuda.amp in train:       {"✅" if "torch.cuda.amp" in train else "❌"}')
print(f'  early_stopping in train:       {"✅" if "early_stopping" in train else "❌"}')
print(f'  terratorch in model:           {"✅" if "terratorch" in model else "❌"}')
print(f'  BACKBONE_REGISTRY in model:    {"✅" if "BACKBONE_REGISTRY" in model else "❌"}')

# ── Step 1: Fix dataset.py pairs_offset (deterministic slice) ─
content = open(dataset_path).read()

if 'pairs_offset' in content:
    print('✅ dataset.py pairs_offset already patched')
else:
    old_init_sig = (
        '        max_pairs=None,\n'
        '        patches_local_dir=None,\n'
        '        features_cache=None,\n'
        '    ):'
    )
    new_init_sig = (
        '        max_pairs=None,\n'
        '        pairs_offset=0,\n'
        '        patches_local_dir=None,\n'
        '        features_cache=None,\n'
        '    ):'
    )
    content = content.replace(old_init_sig, new_init_sig)

    old_sample = (
        '        # Apply max_pairs subsample if specified\n'
        '        if max_pairs and len(self.pairs) > max_pairs:\n'
        '            import random\n'
        '            random.seed(42)\n'
        '            self.pairs = random.sample(self.pairs, max_pairs)\n'
        '        log.info(f"Loaded {len(self.pairs):,} pairs")'
    )
    new_sample = (
        '        # Apply deterministic slice for incremental training\n'
        '        if pairs_offset > 0 or max_pairs:\n'
        '            start = pairs_offset or 0\n'
        '            end   = (start + max_pairs) if max_pairs else len(self.pairs)\n'
        '            self.pairs = self.pairs[start:end]\n'
        '        log.info(f"Loaded {len(self.pairs):,} pairs '
        '(offset={pairs_offset}, max={max_pairs})")'
    )
    content = content.replace(old_sample, new_sample)
    open(dataset_path, 'w').write(content)
    print('✅ dataset.py pairs_offset patched — deterministic slice')

# ── Step 2: Pass pairs_offset to TRAIN dataset only ───────────
content = open(train_path).read()
offset_count = content.count('pairs_offset=cfg.get("pairs_offset", 0)')

if offset_count == 1:
    train_idx  = content.find('train_ds = SentinelPairDataset(')
    val_idx    = content.find('val_ds = SentinelPairDataset(')
    offset_idx = content.find('pairs_offset=cfg.get("pairs_offset", 0)')
    if train_idx < offset_idx < val_idx:
        print('✅ train.py pairs_offset on train dataset only')
    else:
        content = content.replace(
            '        pairs_offset=cfg.get("pairs_offset", 0),\n', ''
        )
        content = content.replace(
            '        features_cache=cfg.get("features_cache", None),\n'
            '    )\n'
            '    val_ds = SentinelPairDataset(',
            '        features_cache=cfg.get("features_cache", None),\n'
            '        pairs_offset=cfg.get("pairs_offset", 0),\n'
            '    )\n'
            '    val_ds = SentinelPairDataset('
        )
        open(train_path, 'w').write(content)
        print('✅ pairs_offset moved to train dataset only')
else:
    content = content.replace(
        '        pairs_offset=cfg.get("pairs_offset", 0),\n', ''
    )
    content = content.replace(
        '        features_cache=cfg.get("features_cache", None),\n'
        '    )\n'
        '    val_ds = SentinelPairDataset(',
        '        features_cache=cfg.get("features_cache", None),\n'
        '        pairs_offset=cfg.get("pairs_offset", 0),\n'
        '    )\n'
        '    val_ds = SentinelPairDataset('
    )
    open(train_path, 'w').write(content)
    count = content.count('pairs_offset=cfg.get("pairs_offset", 0)')
    print(f'✅ train.py pairs_offset on train only — count: {count}'
          if count == 1
          else f'❌ pairs_offset count: {count} — check manually')

# ── Step 3: Add tqdm progress display ─────────────────────────
content = open(train_path).read()

if 'tqdm' in content:
    print('✅ train.py tqdm already added')
else:
    content = content.replace(
        'import mlflow\nimport mlflow.pytorch',
        'import mlflow\nimport mlflow.pytorch\nfrom tqdm import tqdm\nimport sys'
    )
    old_loop = '''    with context:
        for batch_idx, batch in enumerate(loader):'''
    new_loop = '''    with context:
        pbar = tqdm(
            enumerate(loader),
            total=len(loader),
            desc=f"  {'Train' if is_train else 'Val  '} epoch {epoch}",
            ncols=80,
            leave=True,
            file=sys.stdout,
        )
        for batch_idx, batch in pbar:'''
    content = content.replace(old_loop, new_loop)

    old_log_batch = '''            if batch_idx % 50 == 0:
                log.info(
                    f"  {'Train' if is_train else 'Eval'} "
                    f"epoch {epoch} batch {batch_idx}/{n_batches} "
                    f"loss={loss.item():.4f}"
                )'''
    new_log_batch = "            pbar.set_postfix({'loss': f'{loss.item():.4f}'})"
    content = content.replace(old_log_batch, new_log_batch)

    old_avg = '    avg_loss = total_loss / n_batches'
    new_avg = '    print()  # newline after progress bar\n    avg_loss = total_loss / n_batches'
    content = content.replace(old_avg, new_avg)

    old_summary = '''    log.info(
        f"{'Train' if is_train else 'Val'} epoch {epoch} | "
        f"loss={avg_loss:.4f} mae={metrics['mae']:.4f} "
        f"rmse={metrics['rmse']:.4f} r2={metrics['r2']:.4f} "
        f"spearman={metrics['spearman']:.4f} | {duration:.0f}s"
    )'''
    new_summary = '''    prefix = 'Train' if is_train else 'Val  '
    print(
        f"  {prefix} Loss: {avg_loss:.4f} | "
        f"MAE: {metrics['mae']:.4f} | "
        f"R2: {metrics['r2']:.4f} | "
        f"Spearman: {metrics['spearman']:.4f} | "
        f"{duration:.0f}s"
    )
    if not is_train:
        print(f"  {'─'*65}")'''
    content = content.replace(old_summary, new_summary)

    old_epoch = ('    for epoch in range(1, n_epochs + 1):\n'
                 '        epoch_start = time.time()')
    new_epoch  = ('    for epoch in range(1, n_epochs + 1):\n'
                  '        epoch_start = time.time()\n'
                  '        print(f"\\nEpoch {epoch}/{n_epochs} — {phase_name}")')
    content = content.replace(old_epoch, new_epoch)

    open(train_path, 'w').write(content)
    print('✅ train.py tqdm progress display added')

# ── Step 4: Versioned checkpoint filename ─────────────────────
content = open(train_path).read()

if 'best_model_chunk' in content:
    print('✅ train.py versioned checkpoints already wired up')
else:
    old_block = re.compile(
        r"    best_dir\s*=\s*cfg\.get\('checkpoint_best_dir'.*?\)\n"
        r"(?:    chunk_number\s*=\s*cfg\.get\('chunk_number'.*?\)\n)?"
        r"    os\.makedirs\(best_dir.*?\)\n"
        r"    best_path\s*=\s*os\.path\.join\(best_dir,\s*f'best_model_[^']+\.pt'\)",
        re.DOTALL
    )
    new_block = (
        "    best_dir     = cfg.get('checkpoint_best_dir',\n"
        "                       '/gdrive/MyDrive/geoai_mlops/checkpoints/best')\n"
        "    chunk_number = cfg.get('chunk_number', 1)\n"
        "    os.makedirs(best_dir, exist_ok=True)\n"
        '    best_path = os.path.join(best_dir, '
        'f\'best_model_chunk{chunk_number}_r2_{metrics["r2"]:.4f}.pt\')'
    )
    if old_block.search(content):
        content = old_block.sub(new_block, content)
        open(train_path, 'w').write(content)
        print('✅ train.py versioned checkpoints wired up'
              if 'best_model_chunk' in content
              else '❌ Substitution failed')
    else:
        print('❌ Pattern not found')

# ── Step 5: Save best model as dict (consistent format) ───────
content = open(train_path).read()

if 'model_state_dict_or_direct' in content:
    print('✅ train.py best model saved as dict already')
else:
    old_save = "        torch.save(model.state_dict(), best_path)"
    new_save = (
        "        # model_state_dict_or_direct: always save as dict\n"
        "        torch.save({\n"
        "            'model_state_dict': model.state_dict(),\n"
        "            'val_r2':           metrics['r2'],\n"
        "            'chunk_number':     cfg.get('chunk_number', 1),\n"
        "        }, best_path)"
    )
    if old_save in content:
        content = content.replace(old_save, new_save)
        open(train_path, 'w').write(content)
        print('✅ train.py best model saved as dict')
    else:
        if "{'model_state_dict': model.state_dict()" in content:
            content = content.replace(
                "        torch.save({'model_state_dict': model.state_dict(),",
                "        # model_state_dict_or_direct: always save as dict\n"
                "        torch.save({'model_state_dict': model.state_dict(),"
            )
            open(train_path, 'w').write(content)
            print('✅ train.py best model save format confirmed as dict')
        else:
            print('❌ Could not find save line')

# ── Step 6: Metrics log after each chunk ──────────────────────
content = open(train_path).read()

if 'training_log.json' in content:
    print('✅ train.py metrics log already added')
else:
    old_end = '''        log.info(f"\\n✅ Training complete — MLflow run: {run_id}")
        log.info(f"   Best val R²:   {best_val_r2:.4f}")
        log.info(f"   Test R²:       {test_metrics['r2']:.4f}")
        log.info(f"   Test Spearman: {test_metrics['spearman']:.4f}")
        log.info(f"   View results:  http://3.91.55.220:5000")'''

    new_end = '''        log.info(f"\\n✅ Training complete — MLflow run: {run_id}")
        log.info(f"   Best val R²:   {best_val_r2:.4f}")
        log.info(f"   Test R²:       {test_metrics['r2']:.4f}")
        log.info(f"   Test Spearman: {test_metrics['spearman']:.4f}")
        log.info(f"   View results:  http://3.91.55.220:5000")

        # Save metrics log for incremental training tracking
        import json as _json
        log_path     = os.path.join(
            cfg.get('checkpoint_best_dir',
                    '/gdrive/MyDrive/geoai_mlops/checkpoints/best'),
            'training_log.json'
        )
        chunk_number = cfg.get('chunk_number', 1)
        try:
            with open(log_path) as _f:
                training_log = _json.load(_f)
        except (FileNotFoundError, _json.JSONDecodeError):
            training_log = {}
        training_log[f'chunk_{chunk_number}'] = {
            'chunk_number':    chunk_number,
            'pairs_offset':    cfg.get('pairs_offset', 0),
            'max_train_pairs': cfg.get('max_train_pairs', 0),
            'best_val_r2':     round(best_val_r2, 4),
            'test_r2':         round(test_metrics['r2'], 4),
            'test_spearman':   round(test_metrics['spearman'], 4),
            'best_model':      os.path.basename(best_model_path) if best_model_path else None,
            'mlflow_run_id':   run_id,
            'total_chunks':    cfg.get('total_chunks', '?'),
            'is_last_chunk':   cfg.get('is_last_chunk', False),
        }
        with open(log_path, 'w') as _f:
            _json.dump(training_log, _f, indent=2)
        log.info(f'✅ Metrics log saved: {log_path}')
        print(f'\\n✅ Chunk {chunk_number} / {cfg.get("total_chunks","?")} complete!')
        print(f'   Best val R2:   {best_val_r2:.4f}')
        print(f'   Test R2:       {test_metrics["r2"]:.4f}')
        print(f'   Test Spearman: {test_metrics["spearman"]:.4f}')
        print(f'   Best model:    {os.path.basename(best_model_path) if best_model_path else None}')
        if cfg.get('is_last_chunk', False):
            print('\\n🎉 ALL CHUNKS COMPLETE — Final checkpoint saved!')
        else:
            print(f'   Next: set chunk_number={chunk_number+1} in Cell 5')'''

    content = content.replace(old_end, new_end)
    open(train_path, 'w').write(content)
    print('✅ train.py metrics log added')

# ── Step 7: Crash recovery + robust checkpoint loading ────────
content = open(train_path).read()

if 'RESUME_FROM_CHECKPOINT' in content:
    fixes_needed = []
    if "isinstance(ckpt, dict) and 'model_state_dict' in ckpt" not in content:
        fixes_needed.append('format')

    if not fixes_needed:
        print('✅ train.py crash recovery + robust loading already complete')
    else:
        old_crash = re.compile(
            r"                if ckpt_chunk == chunk_number:\n"
            r"                    model\.load_state_dict\(ckpt\['model_state_dict'\]\)\n"
        )
        new_crash = (
            "                if ckpt_chunk == chunk_number:\n"
            "                    if isinstance(ckpt, dict) and 'model_state_dict' in ckpt:\n"
            "                        model.load_state_dict(ckpt['model_state_dict'])\n"
            "                    else:\n"
            "                        model.load_state_dict(ckpt)\n"
        )
        if old_crash.search(content):
            content = old_crash.sub(new_crash, content)

        old_resume = re.compile(
            r"            ckpt = torch\.load\(args\.resume, map_location=device\)\n"
            r"            model\.load_state_dict\(ckpt\['model_state_dict'\]\)\n"
            r"            best_val_r2 = ckpt\.get\('val_r2'.*?\)\n"
        )
        new_resume = (
            "            ckpt = torch.load(args.resume, map_location=device)\n"
            "            if isinstance(ckpt, dict) and 'model_state_dict' in ckpt:\n"
            "                model.load_state_dict(ckpt['model_state_dict'])\n"
            "                best_val_r2 = ckpt.get('val_r2', ckpt.get('best_val_r2', -float('inf')))\n"
            "            else:\n"
            "                model.load_state_dict(ckpt)\n"
            "                best_val_r2 = -float('inf')\n"
        )
        if old_resume.search(content):
            content = old_resume.sub(new_resume, content)

        open(train_path, 'w').write(content)
        print('✅ train.py robust format handling fixed')

else:
    old_start = (
        "        global_step     = 0\n"
        "        best_val_r2     = -float('inf')\n"
        "        best_model_path = None\n"
        "        start_phase     = 1\n"
    )
    new_start = (
        "        global_step     = 0\n"
        "        best_val_r2     = -float('inf')\n"
        "        best_model_path = None\n"
        "        start_phase     = 1\n"
        "        start_epoch_p1  = 1\n"
        "        start_epoch_p2  = 1\n"
        "        chunk_number    = cfg.get('chunk_number', 1)\n"
        "\n"
        "        # ── RESUME_FROM_CHECKPOINT ──────────────────────────────\n"
        "        # Priority 1: crash recovery — latest.pt same chunk\n"
        "        # Priority 2: cross-chunk — --resume best_model_chunkN.pt\n"
        "        # Both handle old (direct state dict) and new (dict) formats\n"
        "\n"
        "        ckpt_dir    = cfg.get('checkpoint_local_dir',\n"
        "                         '/gdrive/MyDrive/geoai_mlops/checkpoints/local')\n"
        "        latest_path = os.path.join(ckpt_dir, 'latest.pt')\n"
        "\n"
        "        if os.path.exists(latest_path):\n"
        "            try:\n"
        "                ckpt       = torch.load(latest_path, map_location=device)\n"
        "                ckpt_chunk = ckpt.get('chunk_number', chunk_number)\n"
        "                if ckpt_chunk == chunk_number:\n"
        "                    if isinstance(ckpt, dict) and 'model_state_dict' in ckpt:\n"
        "                        model.load_state_dict(ckpt['model_state_dict'])\n"
        "                    else:\n"
        "                        model.load_state_dict(ckpt)\n"
        "                    global_step = ckpt.get('step', 0)\n"
        "                    best_val_r2 = ckpt.get('best_val_r2', -float('inf'))\n"
        "                    saved_phase = ckpt.get('phase', 'PHASE_1')\n"
        "                    saved_epoch = ckpt.get('epoch', 1)\n"
        "                    if saved_phase == 'PHASE_1':\n"
        "                        start_phase    = 1\n"
        "                        start_epoch_p1 = saved_epoch + 1\n"
        "                    elif saved_phase == 'PHASE_2':\n"
        "                        start_phase    = 2\n"
        "                        start_epoch_p2 = saved_epoch + 1\n"
        "                    print(f'✅ Crash recovery: resuming {saved_phase} epoch {saved_epoch+1}')\n"
        "                    log.info(f'✅ Crash recovery: {saved_phase} epoch {saved_epoch+1}')\n"
        "                else:\n"
        "                    log.info(f'latest.pt chunk {ckpt_chunk} != {chunk_number} — ignoring')\n"
        "            except Exception as e:\n"
        "                log.warning(f'Could not load latest.pt: {e} — starting fresh')\n"
        "        elif args.resume and os.path.exists(args.resume):\n"
        "            log.info(f'Loading weights from --resume: {args.resume}')\n"
        "            ckpt = torch.load(args.resume, map_location=device)\n"
        "            if isinstance(ckpt, dict) and 'model_state_dict' in ckpt:\n"
        "                model.load_state_dict(ckpt['model_state_dict'])\n"
        "                best_val_r2 = ckpt.get('val_r2', ckpt.get('best_val_r2', -float('inf')))\n"
        "            else:\n"
        "                model.load_state_dict(ckpt)\n"
        "                best_val_r2 = -float('inf')\n"
        "            log.info(f'✅ Weights loaded — best_val_r2: {best_val_r2:.4f}')\n"
        "        elif args.resume:\n"
        "            log.warning(f'Checkpoint not found: {args.resume} — starting fresh')\n"
    )

    if old_start in content:
        content = content.replace(old_start, new_start)

        old_state = (
            "                        'best_val_r2':          best_val_r2,\n"
            "                        'config':               cfg,"
        )
        new_state = (
            "                        'best_val_r2':          best_val_r2,\n"
            "                        'chunk_number':         cfg.get('chunk_number', 1),\n"
            "                        'config':               cfg,"
        )
        content = content.replace(old_state, new_state)

        old_sig = (
            'def train_phase(model, train_loader, val_loader, optimizer, scheduler,\n'
            '                scaler, loss_fn, device, cfg, phase_name, n_epochs,\n'
            '                global_step, best_val_r2, best_model_path,\n'
            '                run_id):'
        )
        new_sig = (
            'def train_phase(model, train_loader, val_loader, optimizer, scheduler,\n'
            '                scaler, loss_fn, device, cfg, phase_name, n_epochs,\n'
            '                global_step, best_val_r2, best_model_path,\n'
            '                run_id, start_epoch=1):'
        )
        content = content.replace(old_sig, new_sig)

        old_epoch_loop = (
            '    for epoch in range(1, n_epochs + 1):\n'
            '        epoch_start = time.time()\n'
            '        print(f"\\nEpoch {epoch}/{n_epochs} — {phase_name}")'
        )
        new_epoch_loop = (
            '    for epoch in range(start_epoch, n_epochs + 1):\n'
            '        epoch_start = time.time()\n'
            '        print(f"\\nEpoch {epoch}/{n_epochs} — {phase_name}")'
        )
        content = content.replace(old_epoch_loop, new_epoch_loop)

        open(train_path, 'w').write(content)
        print('✅ train.py crash recovery wired up')
    else:
        print('❌ start_phase pattern not found — showing context:')
        r = subprocess.run(['grep', '-n', 'start_phase\|global_step',
                           train_path], capture_output=True, text=True)
        print(r.stdout[:500])

# 7e-7g: phase checks and calls
content = open(train_path).read()

if '        if start_phase <= 1:' in content:
    print('✅ train.py phase 1 skip check already updated')
else:
    content = content.replace(
        '        if start_phase == 1:',
        '        if start_phase <= 1:'
    )
    open(train_path, 'w').write(content)
    print('✅ train.py phase 1 skip check updated')

content = open(train_path).read()
if 'start_epoch=start_epoch_p1' in content:
    print('✅ train.py phase 1 call already passes start_epoch')
else:
    old_p1 = (
        "            global_step, best_val_r2, best_model_path = train_phase(\n"
        "                model, train_loader, val_loader,\n"
        "                phase1_optimizer, phase1_scheduler,\n"
        "                scaler, loss_fn, device, cfg,\n"
        "                'PHASE_1', cfg['phase1_epochs'],\n"
        "                global_step, best_val_r2, best_model_path,\n"
        "                run_id\n"
        "            )"
    )
    new_p1 = (
        "            global_step, best_val_r2, best_model_path = train_phase(\n"
        "                model, train_loader, val_loader,\n"
        "                phase1_optimizer, phase1_scheduler,\n"
        "                scaler, loss_fn, device, cfg,\n"
        "                'PHASE_1', cfg['phase1_epochs'],\n"
        "                global_step, best_val_r2, best_model_path,\n"
        "                run_id, start_epoch=start_epoch_p1\n"
        "            )"
    )
    content = content.replace(old_p1, new_p1)
    open(train_path, 'w').write(content)
    print('✅ train.py phase 1 call updated')

content = open(train_path).read()
if 'start_epoch=start_epoch_p2' in content:
    print('✅ train.py phase 2 call already passes start_epoch')
else:
    old_p2 = (
        "        global_step, best_val_r2, best_model_path = train_phase(\n"
        "            model, train_loader, val_loader,\n"
        "            phase2_optimizer, phase2_scheduler,\n"
        "            scaler, loss_fn, device, cfg,\n"
        "            'PHASE_2', cfg['phase2_epochs'],\n"
        "            global_step, best_val_r2, best_model_path,\n"
        "            run_id\n"
        "        )"
    )
    new_p2 = (
        "        global_step, best_val_r2, best_model_path = train_phase(\n"
        "            model, train_loader, val_loader,\n"
        "            phase2_optimizer, phase2_scheduler,\n"
        "            scaler, loss_fn, device, cfg,\n"
        "            'PHASE_2', cfg['phase2_epochs'],\n"
        "            global_step, best_val_r2, best_model_path,\n"
        "            run_id, start_epoch=start_epoch_p2\n"
        "        )"
    )
    content = content.replace(old_p2, new_p2)
    open(train_path, 'w').write(content)
    print('✅ train.py phase 2 call updated')

# ── Step 8: Fix best model loading for test evaluation ────────
content = open(train_path).read()

if 'BEST_MODEL_LOAD_FIX' in content:
    print('✅ train.py best model test loading already fixed')
else:
    old_load = (
        '        # Load best model for test evaluation\n'
        '        if best_model_path and os.path.exists(best_model_path):\n'
        '            model.load_state_dict(\n'
        '                torch.load(best_model_path, map_location=device)\n'
        '            )\n'
        '            log.info("✅ Best model loaded for test evaluation")'
    )
    new_load = (
        '        # Load best model for test evaluation\n'
        '        # BEST_MODEL_LOAD_FIX: handle both dict and direct formats\n'
        '        if best_model_path and os.path.exists(best_model_path):\n'
        '            _ckpt = torch.load(best_model_path, map_location=device)\n'
        "            if isinstance(_ckpt, dict) and 'model_state_dict' in _ckpt:\n"
        "                model.load_state_dict(_ckpt['model_state_dict'])\n"
        '            else:\n'
        '                model.load_state_dict(_ckpt)\n'
        '            log.info("✅ Best model loaded for test evaluation")'
    )
    if old_load in content:
        content = content.replace(old_load, new_load)
        open(train_path, 'w').write(content)
        print('✅ train.py best model test loading fixed')
    else:
        old_load_re = re.compile(
            r"        # Load best model for test evaluation\n"
            r"        if best_model_path and os\.path\.exists\(best_model_path\):\n"
            r"            model\.load_state_dict\(\s*\n"
            r"                torch\.load\(best_model_path.*?\)\s*\n"
            r"            \)\n"
            r'            log\.info\("✅ Best model loaded for test evaluation"\)',
            re.DOTALL
        )
        if old_load_re.search(content):
            content = old_load_re.sub(new_load, content)
            open(train_path, 'w').write(content)
            print('✅ train.py best model test loading fixed (regex)')
        else:
            print('❌ Could not find test load pattern')

# ── Step 9: Best model path recovery when training skipped ────
content = open(train_path).read()

if 'BEST_MODEL_PATH_RECOVERY' in content:
    print('✅ train.py best model path recovery already added')
else:
    old_test_section = (
        '        # Load best model for test evaluation\n'
        '        # BEST_MODEL_LOAD_FIX: handle both dict and direct formats\n'
        '        if best_model_path and os.path.exists(best_model_path):\n'
    )
    new_test_section = (
        '        # Load best model for test evaluation\n'
        '        # BEST_MODEL_LOAD_FIX: handle both dict and direct formats\n'
        '        # BEST_MODEL_PATH_RECOVERY: find best model if path not set\n'
        '        # Handles case where crash recovery skipped all training\n'
        '        if not best_model_path or not os.path.exists(str(best_model_path)):\n'
        '            import glob as _glob\n'
        '            _best_dir  = cfg.get("checkpoint_best_dir",\n'
        '                             "/gdrive/MyDrive/geoai_mlops/checkpoints/best")\n'
        '            _chunk_num = cfg.get("chunk_number", 1)\n'
        '            _pattern   = os.path.join(_best_dir,\n'
        '                             f"best_model_chunk{_chunk_num}_r2_*.pt")\n'
        '            _candidates = _glob.glob(_pattern)\n'
        '            if _candidates:\n'
        '                best_model_path = max(_candidates,\n'
        '                    key=lambda x: float(\n'
        '                        x.split("_r2_")[1].replace(".pt", "")))\n'
        '                log.info(f"✅ Best model path recovered: {best_model_path}")\n'
        '                print(f"✅ Best model path recovered: "\n'
        '                      f"{os.path.basename(best_model_path)}")\n'
        '            else:\n'
        '                log.warning(\n'
        '                    f"No best model found for chunk {_chunk_num} "\n'
        '                    f"in {_best_dir} — test eval will use current weights")\n'
        '        if best_model_path and os.path.exists(best_model_path):\n'
    )
    if old_test_section in content:
        content = content.replace(old_test_section, new_test_section)
        open(train_path, 'w').write(content)
        print('✅ train.py best model path recovery added')
    else:
        old_test_re = re.compile(
            r"        # Load best model for test evaluation\n"
            r"        # BEST_MODEL_LOAD_FIX.*?\n"
            r"        if best_model_path and os\.path\.exists\(best_model_path\):\n",
            re.DOTALL
        )
        if old_test_re.search(content):
            content = old_test_re.sub(new_test_section, content)
            open(train_path, 'w').write(content)
            print('✅ train.py best model path recovery added (regex)')
        else:
            print('❌ Could not find test section')

# ── Step 10: tqdm + GPU cache clear for test evaluation ───────
content = open(train_path).read()

if 'TEST_EVAL_TQDM' in content:
    print('✅ train.py test evaluation tqdm already added')
else:
    old_eval = (
        'def evaluate_test(model, test_loader, loss_fn, device, cfg):\n'
        '    """Final evaluation on Brandenburg test set."""\n'
        '    log.info("\\n" + "="*60)\n'
        '    log.info("FINAL TEST EVALUATION (Brandenburg)")\n'
        '    log.info("="*60)\n'
        '\n'
        '    model.eval()\n'
        '    all_preds   = []\n'
        '    all_targets = []\n'
        '\n'
        '    with torch.no_grad():\n'
        '        for batch in test_loader:\n'
    )
    new_eval = (
        'def evaluate_test(model, test_loader, loss_fn, device, cfg):\n'
        '    """Final evaluation on Brandenburg test set."""\n'
        '    log.info("\\n" + "="*60)\n'
        '    log.info("FINAL TEST EVALUATION (Brandenburg)")\n'
        '    log.info("="*60)\n'
        '\n'
        '    # TEST_EVAL_TQDM: clear GPU cache + show progress\n'
        '    if torch.cuda.is_available():\n'
        '        torch.cuda.empty_cache()\n'
        '        log.info("✅ GPU cache cleared before test evaluation")\n'
        '\n'
        '    model.eval()\n'
        '    all_preds   = []\n'
        '    all_targets = []\n'
        '    n_batches   = len(test_loader)\n'
        '\n'
        '    print(f"\\nTest evaluation: {n_batches} batches")\n'
        '    with torch.no_grad():\n'
        '        pbar = tqdm(test_loader, total=n_batches,\n'
        '                    desc="  Test ", ncols=80,\n'
        '                    leave=True, file=sys.stdout)\n'
        '        for batch in pbar:\n'
    )
    if old_eval in content:
        content = content.replace(old_eval, new_eval)
        open(train_path, 'w').write(content)
        print('✅ train.py test evaluation tqdm + GPU cache clear added')
    else:
        old_eval_re = re.compile(
            r"def evaluate_test\(model, test_loader, loss_fn, device, cfg\):\n"
            r'    """Final evaluation on Brandenburg test set."""\n'
            r"    log\.info.*?\n"
            r"    log\.info.*?\n"
            r"    log\.info.*?\n"
            r"\n"
            r"    model\.eval\(\)\n"
            r"    all_preds\s*=\s*\[\]\n"
            r"    all_targets\s*=\s*\[\]\n"
            r"\n"
            r"    with torch\.no_grad\(\):\n"
            r"        for batch in test_loader:\n",
            re.DOTALL
        )
        if old_eval_re.search(content):
            content = old_eval_re.sub(new_eval, content)
            open(train_path, 'w').write(content)
            print('✅ train.py test evaluation tqdm added (regex)')
        else:
            print('❌ Could not find evaluate_test function')

# ── Pre-flight test ────────────────────────────────────────────
content = open(train_path).read()
lines   = content.splitlines()
in_save_best = False
chunk_found  = False
path_found   = False

for line in lines:
    if 'def save_best_model' in line:
        in_save_best = True
    if in_save_best:
        if 'chunk_number' in line and 'cfg.get' in line:
            chunk_found = True
        if 'best_model_chunk' in line:
            path_found = True
        if path_found:
            break

print()
print('Pre-flight test — save_best_model function:')
print(f'  chunk_number defined:  {"✅" if chunk_found else "❌ WILL FAIL AT RUNTIME"}')
print(f'  versioned path:        {"✅" if path_found else "❌ WILL FAIL AT RUNTIME"}')

if not chunk_found or not path_found:
    print('❌ STOP — do not run Cell 8')
else:
    offset_count = content.count('pairs_offset=cfg.get("pairs_offset", 0)')
    print()
    print('Final checks:')
    ds_content = open(dataset_path).read()
    print(f'  pairs_offset in dataset:     {"✅" if "pairs_offset" in ds_content else "❌"}')
    print(f'  pairs_offset train only:     {"✅" if offset_count == 1 else f"❌ found {offset_count} times"}')
    print(f'  tqdm in train:               {"✅" if "tqdm" in content else "❌"}')
    print(f'  versioned checkpoints:       {"✅" if "best_model_chunk" in content else "❌"}')
    print(f'  best model saved as dict:    {"✅" if "model_state_dict_or_direct" in content else "❌"}')
    print(f'  metrics log:                 {"✅" if "training_log.json" in content else "❌"}')
    print(f'  crash recovery:              {"✅" if "RESUME_FROM_CHECKPOINT" in content else "❌"}')
    print(f'  robust format handling:      {"✅" if "isinstance(ckpt, dict) and" in content else "❌"}')
    print(f'  start_epoch_p1:              {"✅" if "start_epoch_p1" in content else "❌"}')
    print(f'  start_epoch_p2:              {"✅" if "start_epoch_p2" in content else "❌"}')
    print(f'  phase 1 skip check:          {"✅" if "start_phase <= 1" in content else "❌"}')
    print(f'  phase 1 start_epoch:         {"✅" if "start_epoch=start_epoch_p1" in content else "❌"}')
    print(f'  phase 2 start_epoch:         {"✅" if "start_epoch=start_epoch_p2" in content else "❌"}')
    print(f'  test load fix:               {"✅" if "BEST_MODEL_LOAD_FIX" in content else "❌"}')
    print(f'  best model path recovery:    {"✅" if "BEST_MODEL_PATH_RECOVERY" in content else "❌"}')
    print(f'  test eval tqdm:              {"✅" if "TEST_EVAL_TQDM" in content else "❌"}')
    print()
    print('✅ Cell 7 complete — ready for Cell 8')

In [ ]:
# ============================================================
# CELL 8 — Run Training (incremental with versioned checkpoints)
# ============================================================
import subprocess, os, json, glob, torch
from google.colab import userdata

VENV_PYTHON = '/content/venv/bin/python3'
AWS_KEY     = userdata.get('AWS_ACCESS_KEY_ID')
AWS_SECRET  = userdata.get('AWS_SECRET_ACCESS_KEY')

env = {
    **os.environ,
    'MPLBACKEND':            'agg',
    'AWS_ACCESS_KEY_ID':     AWS_KEY,
    'AWS_SECRET_ACCESS_KEY': AWS_SECRET,
    'AWS_DEFAULT_REGION':    'us-east-1',
}

# Read config
cfg          = json.load(open('/content/geoai-mlops-p1/src/training/train_config.json'))
chunk_number = cfg.get('chunk_number', 1)
total_chunks = cfg.get('total_chunks', '?')
best_dir     = cfg.get('checkpoint_best_dir',
                       '/gdrive/MyDrive/geoai_mlops/checkpoints/best')
ckpt_dir     = cfg.get('checkpoint_local_dir',
                       '/gdrive/MyDrive/geoai_mlops/checkpoints/local')
latest_path  = os.path.join(ckpt_dir, 'latest.pt')
pairs_offset = cfg.get('pairs_offset', 0)
max_train    = cfg.get('max_train_pairs', 0)
is_last      = cfg.get('is_last_chunk', False)

# Guard — check if all data already trained
if pairs_offset >= cfg.get('max_train_pairs', 0) and chunk_number > total_chunks:
    print('✅ All chunks complete — nothing left to train')
    print(f'   Final checkpoint in: {best_dir}')
    raise SystemExit('Training complete')

# Print session summary
print(f'{"═"*60}')
print(f'  CHUNK {chunk_number} / {total_chunks}')
print(f'  Train pairs: {pairs_offset:,} → {pairs_offset+max_train:,} ({max_train:,} pairs)')
print(f'  {"⚠️  LAST CHUNK" if is_last else f"Chunks remaining: {total_chunks-chunk_number}"}')
print(f'{"═"*60}')

# Show existing training log if available
log_path = os.path.join(best_dir, 'training_log.json')
if os.path.exists(log_path):
    print('\nPrevious chunks:')
    training_log = json.load(open(log_path))
    for key, val in sorted(training_log.items()):
        print(f'  {key}: val_r2={val["best_val_r2"]:.4f} '
              f'| pairs={val["pairs_offset"]:,}→'
              f'{val["pairs_offset"]+val["max_train_pairs"]:,} '
              f'| {val["best_model"]}')
    print()

# ── Manage latest.pt for crash recovery ───────────────────────
if os.path.exists(latest_path):
    try:
        ckpt       = torch.load(latest_path, map_location='cpu')
        ckpt_chunk = ckpt.get('chunk_number', None)
        if ckpt_chunk == chunk_number:
            saved_phase = ckpt.get('phase', 'PHASE_1')
            saved_epoch = ckpt.get('epoch', 1)
            print(f'✅ latest.pt found for chunk {chunk_number}')
            print(f'   Will resume: {saved_phase} from epoch {saved_epoch + 1}')
        else:
            os.remove(latest_path)
            print(f'🗑️  Cleared latest.pt (belonged to chunk {ckpt_chunk}, now chunk {chunk_number})')
    except Exception as e:
        os.remove(latest_path)
        print(f'🗑️  Cleared corrupted latest.pt: {e}')
else:
    print(f'ℹ️  No latest.pt found — starting fresh for chunk {chunk_number}')

# ── Build command ──────────────────────────────────────────────
cmd = [VENV_PYTHON, 'src/training/train.py',
       '--config', 'src/training/train_config.json']

# Add --resume if chunk > 1 (cross-chunk weights)
if chunk_number > 1:
    prev_chunk  = chunk_number - 1
    pattern     = os.path.join(best_dir, f'best_model_chunk{prev_chunk}_r2_*.pt')
    candidates  = glob.glob(pattern)
    if candidates:
        resume_path = max(
            candidates,
            key=lambda x: float(x.split('_r2_')[1].replace('.pt', ''))
        )
        cmd += ['--resume', resume_path]
        print(f'✅ Cross-chunk resume: {os.path.basename(resume_path)}')
    else:
        print(f'⚠️  No checkpoint found for chunk {prev_chunk} — starting fresh')
else:
    print(f'✅ Chunk 1 — starting fresh (no cross-chunk resume)')

print(f'\n   Checkpoint will save as:')
print(f'   best_model_chunk{chunk_number}_r2_<val_r2>.pt')
print()

# Stream output line by line
process = subprocess.Popen(
    cmd,
    cwd='/content/geoai-mlops-p1',
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

for line in process.stdout:
    print(line, end='', flush=True)

process.wait()

print(f'\nReturn code: {process.returncode}')

if process.returncode == 0:
    if os.path.exists(log_path):
        training_log = json.load(open(log_path))
        chunk_key    = f'chunk_{chunk_number}'
        if chunk_key in training_log:
            entry = training_log[chunk_key]
            print(f'\n{"═"*60}')
            print(f'  CHUNK {chunk_number} COMPLETE')
            print(f'  Best val R2:  {entry["best_val_r2"]:.4f}')
            print(f'  Best model:   {entry["best_model"]}')
            print(f'  MLflow run:   {entry["mlflow_run_id"]}')
            print(f'{"═"*60}')

    if is_last:
        print('\n🎉 ALL CHUNKS COMPLETE!')
        print(f'   Final checkpoint saved in: {best_dir}')
        print(f'   View full history: {log_path}')
    else:
        print(f'\n   Next: set chunk_number={chunk_number+1} in Cell 5')

    print(f'   View results at: http://3.91.55.220:5000')
